In [ ]:
import pandas as pd
import re
from urllib.parse import urlparse


SUSPICIOUS_KEYWORDS = [
    "login", "secure", "account", "update", "verify", "bank",
    "free", "confirm", "password", "wp-admin"
]

SUSPICIOUS_TLDS = [
    ".tk", ".ml", ".ga", ".cf", ".gq", ".xyz"
]

SPECIAL_CHARS = r"[@\?\-=%&:/#\.]"

def has_ip(url):
    return int(bool(re.search(r"\b\d{1,3}(\.\d{1,3}){3}\b", url)))

def count_special_chars(url):
    return len(re.findall(SPECIAL_CHARS, url))

def num_subdomains(url):
    try:
        domain = urlparse(url).netloc
        if domain.startswith("www."):
            domain = domain[4:]
        return max(domain.count(".") - 1, 0)
    except:
        return 0

def suspicious_keywords(url):
    url = url.lower()
    return int(any(keyword in url for keyword in SUSPICIOUS_KEYWORDS))

def suspicious_tld(url):
    return int(any(url.lower().endswith(tld) for tld in SUSPICIOUS_TLDS))

# -------------------------
# Main preprocessing
# -------------------------
def preprocess(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    # Rename column
    df.rename(columns={"type": "label"}, inplace=True)

    # Feature engineering
    df["URL_LENGTH"] = df["url"].apply(len)
    df["NUMBER_SPECIAL_CHARACTERS"] = df["url"].apply(count_special_chars)
    df["HAS_IP"] = df["url"].apply(has_ip)
    df["HAS_AT"] = df["url"].apply(lambda x: int("@" in x))
    df["HAS_HTTPS"] = df["url"].apply(lambda x: int(x.startswith("https")))
    df["NUM_SUBDOMAINS"] = df["url"].apply(num_subdomains)
    df["NUM_DIGITS"] = df["url"].apply(lambda x: sum(c.isdigit() for c in x))
    df["SUSPICIOUS_KEYWORDS"] = df["url"].apply(suspicious_keywords)
    df["SUSPICIOUS_TLD"] = df["url"].apply(suspicious_tld)

    # Reorder columns
    df = df[
        [
            "url",
            "label",
            "URL_LENGTH",
            "NUMBER_SPECIAL_CHARACTERS",
            "HAS_IP",
            "HAS_AT",
            "HAS_HTTPS",
            "NUM_SUBDOMAINS",
            "NUM_DIGITS",
            "SUSPICIOUS_KEYWORDS",
            "SUSPICIOUS_TLD",
        ]
    ]

    # -------------------------
    # Balance classes by deletion (undersampling)
    # -------------------------
    min_size = df["label"].value_counts().min()

    df_balanced = (
        df.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(min_size, random_state=42))
        .reset_index(drop=True)
    )

    # Save result
    df_balanced.to_csv(output_csv, index=False)

    print("Preprocessing complete")
    print(df_balanced["label"].value_counts())

# -------------------------
# Run
# -------------------------
if __name__ == "__main__":
    preprocess("input_urls.csv", "processed_balanced_urls.csv")
